# T2ICount Colab bootstrap
This notebook only orchestrates repository setup, validation, and training commands. Model, loss, and inference logic remain in repository modules.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ASSET_ROOT = "/content/drive/MyDrive/T2ICount-assets"
REPO_DIR = "/content/T2ICount-Implementation"

In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/T2ICount-assets.zip")
ASSET_ROOT_PATH = Path(ASSET_ROOT)

if ASSET_ROOT_PATH.is_dir():
    print("Assets already extracted:", ASSET_ROOT_PATH)

elif not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Cannot find {ZIP_PATH}")

else:
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        names = [name.replace("\\", "/") for name in zf.namelist()]

        has_root_folder = any(
            name.startswith("T2ICount-assets/")
            for name in names
        )

        if has_root_folder:
            extract_to = ASSET_ROOT_PATH.parent
        else:
            ASSET_ROOT_PATH.mkdir(parents=True, exist_ok=True)
            extract_to = ASSET_ROOT_PATH

        print("Extracting to:", extract_to)
        zf.extractall(extract_to)

    print("Extraction complete.")

## 2. Clone or update the repository

In [ ]:
import os
import pathlib
import subprocess

REPO_URL = "https://github.com/bqa100507-spec/T2ICount-Implementation.git"
repo_path = pathlib.Path(REPO_DIR)
if not repo_path.exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
elif not (repo_path / ".git").is_dir():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
print("Repository:", pathlib.Path.cwd())

## 3. Install Colab dependencies
If installation replaces important runtime packages such as PyTorch, Transformers, or Lightning, restart the runtime/kernel before continuing when required. Do not continue validation with a partially reloaded environment.

In [ ]:
%pip install -r requirements-colab.txt

## 4. Configure the notebook Python environment

In [ ]:
import os

os.environ["T2ICOUNT_ASSET_ROOT"] = ASSET_ROOT
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("T2ICOUNT_ASSET_ROOT=", os.environ["T2ICOUNT_ASSET_ROOT"] )

## 5. Validate Drive assets and offline CLIP loading

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "scripts/check_assets.py",
    "--asset-root", ASSET_ROOT,
    "--data", "fsc147",
    "--check-offline-load",
], check=True)

## 6. Numerical smoke test
Known reference: image `2`, prompt `sea shells`, GT `8`, prediction approximately `8.165655`. A small difference across CUDA/PyTorch versions is acceptable; a large difference is not. Do not start training if this check is wrong.

In [ ]:
subprocess.run([
    sys.executable, "test.py",
    "--asset-root", ASSET_ROOT,
    "--data", "fsc147",
    "--batch-size", "1",
    "--max-samples", "1",
], check=True)

## 7. VSCode terminal environment
A VSCode integrated terminal is a separate shell; variables set with `os.environ` in this notebook kernel are not guaranteed to exist there. Run the following in each VSCode/Colab terminal session before launching commands:

```bash
cd /content/T2ICount-Implementation
export T2ICOUNT_ASSET_ROOT="/content/drive/MyDrive/T2ICount-assets"
export HF_HUB_OFFLINE=1
export TRANSFORMERS_OFFLINE=1
```

## 8. Mini training smoke run (intentional)
Run this cell only after both checks above pass. It uses the existing defaults except for one epoch and verifies loading, initialization, forward/loss/backward, an optimizer step, and checkpoint/log persistence to Drive. Outputs are written below `checkpoints/baseline_retrain/smoke/baseline_smoke/`.

In [ ]:
subprocess.run([
    sys.executable, "train.py",
    "--asset-root", ASSET_ROOT,
    "--save-dir", f"{ASSET_ROOT}/checkpoints/baseline_retrain/smoke",
    "--content", "baseline_smoke",
    "--epochs", "1",
], check=True)

## 9. Full baseline training (documented, not started)
The command below keeps the established training defaults and persists the run at `/content/drive/MyDrive/T2ICount-assets/checkpoints/baseline_retrain/run_01/`. Uncomment it only when ready. Resume requires a full-state `.tar` checkpoint, not a weights-only `.pth` file.

```bash
python train.py \
  --asset-root "/content/drive/MyDrive/T2ICount-assets" \
  --save-dir "/content/drive/MyDrive/T2ICount-assets/checkpoints/baseline_retrain" \
  --content run_01

# After an interruption, add for example:
# --resume "/content/drive/MyDrive/T2ICount-assets/checkpoints/baseline_retrain/run_01/50_ckpt.tar"
```